In [12]:
import trimesh as tri
import open3d as o3d
import numpy as np
import skimage as ski
import pyvista as pv
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import copy
import gc
import time

### Visualize One Mesh and Point Cloud Overlaid

In [2]:

def show_queries(pointcloud_file, mesh_file):
    points = np.load(pointcloud_file)
    point_cloud = o3d.geometry.PointCloud()
    point_cloud.points = o3d.utility.Vector3dVector(points)
    point_cloud.paint_uniform_color([1.0, 0.1, 0.1])
    mesh = o3d.io.read_triangle_mesh(mesh_file)
    o3d.visualization.draw_plotly([point_cloud, mesh], width=900, height=700)

show_queries("00010218_4769314c71814669ba5d3512_trimesh_013.xyz.npy", "00010218_4769314c71814669ba5d3512_trimesh_013.ply")


### Mesh Metrics

In [2]:

class mesh_metrics:
  @staticmethod
  def extract_component(mesh, rank=0):
    """Extract component by descending surface area."""
    import copy
    import numpy as np

    labels, sizes, areas = mesh.cluster_connected_triangles()

    labels = np.asarray(labels)
    areas = np.asarray(areas)

    ordered_ids = np.argsort(areas)[::-1]

    if rank >= len(ordered_ids):
        return None

    component_id = ordered_ids[rank]

    component = copy.deepcopy(mesh)
    component.remove_triangles_by_mask(labels != component_id)
    component.remove_unreferenced_vertices()

    return component

  # @staticmethod
  # def check_edges(mesh):
  #   mesh = mesh.remove_duplicated_triangles()
  #   triangles = np.asarray(mesh.triangles)
  #   edges = np.vstack([
  #       triangles[:, [0, 1]],
  #       triangles[:, [1, 2]],
  #       triangles[:, [2, 0]],
  #   ])

  #   # Sort vertices on each edge so direction doesn't mask duplicates (e.g., [1,2] and [2,1] match)
  #   sorted_edges = np.sort(edges, axis=1)

  #   # Count how many times each unique edge appears in the mesh
  #   unique_edges, counts = np.unique(sorted_edges, axis=0, return_counts=True)

  #   # Interpret the counts:
  #   num_boundaries = np.sum(counts == 1)
  #   num_interior = np.sum(counts == 2)
  #   num_non_manifold = np.sum(counts >= 3)
  #   return num_boundaries, num_interior, num_non_manifold

  def __init__(self, mesh):
    """
      Args:
        mesh: Open3D TriangleMesh Object
    """
    self.mesh = copy.deepcopy(mesh)
    self.n_vertices = len(mesh.vertices)
    self.n_triangles = len(mesh.triangles)
    self.genus = None
    self.topologically_watertight = None
    self.o3d_watertight = None

    self.orientable = None
    self.self_intersecting = None
    self.edge_manifold = None
    self.vertex_manifold = None

    total_area = mesh.get_surface_area()
    ## Component, genus, and area checking
    _, cluster_sizes, _ = mesh.cluster_connected_triangles()
    self.n_components = len(cluster_sizes)

    # Largest cc Metrics
    if not (self.n_components == 1):
      largest_cc = self.extract_component(mesh, rank=0)
    else:
      largest_cc = mesh

    self.largest_cc_area = largest_cc.get_surface_area()
    self.largest_cc_area_percentage = self.largest_cc_area/total_area
    self.largest_cc_o3d_watertight = largest_cc.is_watertight()
    self.largest_cc_orientable = largest_cc.is_orientable()
    self.largest_cc_self_intersecting = largest_cc.is_self_intersecting()
    self.largest_cc_edge_manifold_closed = largest_cc.is_edge_manifold(allow_boundary_edges=False)
    self.largest_cc_edge_manifold_open = largest_cc.is_edge_manifold(allow_boundary_edges=True) and not largest_cc.is_edge_manifold(allow_boundary_edges=False)
    self.largest_cc_vertex_manifold = largest_cc.is_vertex_manifold()

    if self.largest_cc_edge_manifold_closed:
      self.largest_cc_genus = (2 - largest_cc.euler_poincare_characteristic()) / 2
    elif self.largest_cc_edge_manifold_open: #check connected components
      largest_cc_he = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(largest_cc)
      self.largest_cc_n_boundary_components = len(largest_cc_he.get_boundaries())
      self.largest_cc_genus = (2 - largest_cc.euler_poincare_characteristic() - self.largest_cc_n_boundary_components) / 2
    else:
      self.largest_cc_genus = None

    # Second-Largest cc Metrics
    if not (self.n_components == 1):
      second_cc = self.extract_component(mesh, rank=1)
      self.second_cc_area = second_cc.get_surface_area()
      self.second_cc_area_percentage = self.second_cc_area/total_area
      self.second_cc_o3d_watertight = second_cc.is_watertight()
      self.second_cc_edge_manifold = second_cc.is_edge_manifold()
      # if self.second_cc_edge_manifold:
      #   self.second_cc_genus = (2 - second_cc.euler_poincare_characteristic()) / 2
      # else:
      #   second_cc_he = o3d.geometry.HalfEdgeTriangleMesh.create_from_triangle_mesh(second_cc)
      #   self.second_cc_n_boundary_components = len(second_cc_he.get_boundaries())
      #   self.second_cc_genus = (2 - second_cc_genus.euler_poincare_characteristic() - self.second_cc_n_boundary_components) / 2

    self.genus = self.largest_cc_genus
    self.o3d_watertight = self.largest_cc_o3d_watertight
    self.orientable = self.largest_cc_orientable
    self.self_intersecting = self.largest_cc_self_intersecting
    self.edge_manifold_open = self.largest_cc_edge_manifold_open
    self.edge_manifold_closed = self.largest_cc_edge_manifold_closed
    self.vertex_manifold = self.largest_cc_vertex_manifold
    self.topologically_watertight = self.edge_manifold

    return
  
  def __str__(self):
      lines = [f"{self.__class__.__name__}:"]
      for k, v in self.__dict__.items():
          lines.append(f"  {k}: {v}")
      return "\n".join(lines)

# Assume actual mesh as input
def chamfer_L2_squared_point2point(pred_mesh, gt_mesh, n_samples=10000):
  pred_points = pred_mesh.sample_points_uniformly(n_samples)
  gt_points = gt_mesh.sample_points_uniformly(n_samples)
  d_pred2gt = np.asarray(pred_points.compute_point_cloud_distance(gt_points))
  d_gt2pred = np.asarray(gt_points.compute_point_cloud_distance(pred_points))
  return (np.mean(d_pred2gt**2) + np.mean(d_gt2pred**2))

# assume mesh_metrics as input
def mesh2mesh_metrics(pred_mesh, gt_mesh):

  # Built-in o3d metrics
  params = o3d.t.geometry.MetricParameters(
    n_sampled_points=10000,
    fscore_radius=np.array([
        0.005,
        0.01,
        0.02,
    ])
  )
  pred_mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(pred_mesh.mesh)
  gt_mesh_t = o3d.t.geometry.TriangleMesh.from_legacy(gt_mesh.mesh)
  # might need to be tensor?
  result = pred_mesh_t.compute_metrics(
      gt_mesh_t,
      (
          o3d.t.geometry.Metric.ChamferDistance,
          o3d.t.geometry.Metric.HausdorffDistance,
          o3d.t.geometry.Metric.FScore,
      ),
      params,
  )
  chamfer_L2_point2surface = result[0]
  Hausdorff = result[1]
  F_point2surface_005 = result[2]
  F_point2surface_01 = result[3]
  F_point2surface_02 = result[4]

  chamfer_L2_sqrd_p2p = chamfer_L2_squared_point2point(pred_mesh.mesh, gt_mesh.mesh, 10000)

  return {
    "chamfer_L2_p2s": chamfer_L2_point2surface,
    "chamfer_L2_sqrd_p2p": chamfer_L2_sqrd_p2p,
    "F_p2s_005": F_point2surface_005,
    "F_p2s": F_point2surface_01,
    "F_p2s_02": F_point2surface_02,
    "genus consistency": (pred_mesh.genus == gt_mesh.genus),
    "watertight consistency": (pred_mesh.topologically_watertight == gt_mesh.topologically_watertight)
  }


mesh = o3d.io.read_triangle_mesh("00010218_4769314c71814669ba5d3512_trimesh_013.ply")
pairs = np.asarray(mesh.get_self_intersecting_triangles())
bad_ids = np.unique(pairs)
bad_mesh = o3d.geometry.TriangleMesh(
    vertices=o3d.utility.Vector3dVector(
        np.asarray(mesh.vertices).copy()
    ),
    triangles=o3d.utility.Vector3iVector(
        np.asarray(mesh.triangles)[bad_ids].copy()
    )
)
bad_mesh.remove_unreferenced_vertices()
bad_mesh.compute_vertex_normals()



armadillo = o3d.io.read_triangle_mesh(o3d.data.ArmadilloMesh().path)
cube = o3d.geometry.TriangleMesh.create_box(width=1.0, height=1.0, depth=1.0)
#metrics1 = mesh_metrics(cube)
metrics2 = mesh_metrics(mesh)
#print(metrics1)
print(metrics2)
print(mesh2mesh_metrics(metrics2, metrics2))
o3d.visualization.draw_plotly([metrics2.mesh], width=900, height=700)
#o3d.visualization.draw_plotly([bad_mesh], width=900, height=700)



mesh_metrics:
  mesh: TriangleMesh with 2142 points and 4296 triangles.
  n_vertices: 2142
  n_triangles: 4296
  genus: 4.0
  topologically_watertight: None
  o3d_watertight: False
  orientable: True
  self_intersecting: True
  edge_manifold: None
  vertex_manifold: True
  n_components: 1
  largest_cc_area: 2.729117256339001
  largest_cc_area_percentage: 1.0
  largest_cc_o3d_watertight: False
  largest_cc_orientable: True
  largest_cc_self_intersecting: True
  largest_cc_edge_manifold_closed: True
  largest_cc_edge_manifold_open: False
  largest_cc_vertex_manifold: True
  largest_cc_genus: 4.0
  edge_manifold_open: False
  edge_manifold_closed: True
{'chamfer_L2_p2s': 1.6183312e-06
Tensor[shape={}, stride={}, Float32, CPU:0, 0x5aab1c94a490], 'chamfer_L2_sqrd_p2p': np.float64(0.00017101481629951516), 'F_p2s_005': 100
Tensor[shape={}, stride={}, Float32, CPU:0, 0x5aab1c94a498], 'F_p2s': 100
Tensor[shape={}, stride={}, Float32, CPU:0, 0x5aab1c94a49c], 'F_p2s_02': 100
Tensor[shape={}, stri

### Screened Poisson Reconstruction (SPR)

In [8]:
def SPR(point_cloud, depth=9, trim=True, clock=True):

    gc.collect()
    t_start = time.perf_counter()
    point_cloud.estimate_normals(search_param=o3d.geometry.KDTreeSearchParamHybrid(radius=0.1, max_nn=30))
    point_cloud.orient_normals_consistent_tangent_plane(k=15)
    mesh, densities = o3d.geometry.TriangleMesh.create_from_point_cloud_poisson(
        point_cloud, depth=depth, width=0, scale=1.1, linear_fit=False
    )
    removed_vertices = densities < np.quantile(densities, 0.05)
    mesh.remove_vertices_by_mask(removed_vertices)

    t_end = time.perf_counter()
    if clock:
        return mesh, (t_end - t_start)
        
    return mesh

test_pcd = "00010218_4769314c71814669ba5d3512_trimesh_013.xyz.npy"
#test = SPR(test_pcd)
gt = o3d.io.read_triangle_mesh("00010218_4769314c71814669ba5d3512_trimesh_013.ply")


o3d.visualization.draw_plotly([gt], width=900, height=700)
o3d.visualization.draw_plotly([test], width=900, height=700)

NameError: name 'test' is not defined

### Test Marching Cubes

In [ ]:
def scalar_to_blue_red(arr):
  # Normalize array to [0, 1]
  vmin, vmax = np.min(arr), np.max(arr)
  if vmax == vmin:
    t = np.zeros_like(arr, dtype=float)
  else:
    t = (arr - vmin) / (vmax - vmin)

  # Shape of t: (H, W) -> expand to (H, W, 1) for broadcasting
  t = t[..., np.newaxis]

  # Blue is (0, 0, 1), Red is (1, 0, 0)
  blue = np.array([0.0, 0.0, 1.0])
  red = np.array([1.0, 0.0, 0.0])

  # Linear interpolation: (1 - t)*blue + t*red
  rgb = (1.0 - t) * blue + t * red
  return rgb


def marching_cubes(mesh=None, grid_length=128):
  if not mesh:
    mesh = o3d.t.geometry.TriangleMesh.from_legacy(o3d.data.ArmadilloMesh().path)
  
  # RaycastingScene() requires tensor format
  # Create a scene and add the triangle mesh
  scene = o3d.t.geometry.RaycastingScene()
  scene.add_triangles(mesh)  # we do not need the geometry ID for mesh
  min_bound = mesh.vertex.positions.min(0).numpy()
  max_bound = mesh.vertex.positions.max(0).numpy()
  xyz_range, step = np.linspace(min_bound, max_bound, num=grid_length, retstep=True)

  # query_points is a [n,n,n,3] array ..
  query_points = np.stack(np.meshgrid(*xyz_range.T, indexing='ij'), axis=-1).astype(np.float32)

  # signed distance is a [32,32,32] array
  sdf = scene.compute_signed_distance(query_points).numpy()
  verts, faces, normals, values = ski.measure.marching_cubes(np.asarray(sdf), 0, gradient_direction='ascent')
  # Return to scale of original pointcloud
  verts = (verts * step) + min_bound

  colors = scalar_to_blue_red(values)

  # Prepare o3d mesh for rendering (nice viewer)
  mesh = o3d.geometry.TriangleMesh()
  mesh.vertices = o3d.utility.Vector3dVector(verts)
  mesh.vertex_colors = o3d.utility.Vector3dVector(colors)
  mesh.triangles = o3d.utility.Vector3iVector(faces)
  mesh.vertex_normals = o3d.utility.Vector3dVector(normals)

  # Prepare trimesh mesh for rendering (permits vertex colors)
  # Coloration visualized distance sdf's 0 level-set to each vertex, red is further and shade is relative per mesh (therefore less distance across the shape won't necesarrily make it less red).
  mesh_tri = tri.Trimesh(vertices=verts, faces=faces, vertex_normals=normals, vertex_colors=colors)
  tri.repair.fix_normals(mesh_tri)
  #mesh_tri.show()
  return mesh, mesh_tri


servo_mesh = o3d.t.geometry.TriangleMesh.from_legacy(o3d.io.read_triangle_mesh("00010218_4769314c71814669ba5d3512_trimesh_013.ply"))
servo_points = np.load("00010218_4769314c71814669ba5d3512_trimesh_013.xyz.npy")
servo_pcd = o3d.geometry.PointCloud()
servo_pcd.points = o3d.utility.Vector3dVector(servo_points)


gt_mesh = servo_mesh
input_pcd = servo_pcd

oracle_mesh, _ = marching_cubes(mesh=gt_mesh)
recon_mesh, spr_time = SPR(input_pcd, clock=True)
#Convert for visualization and metrics
gt_mesh = gt_mesh.to_legacy()

o3d.visualization.draw_plotly([gt_mesh], width=900, height=700)
o3d.visualization.draw_plotly([oracle_mesh], width=900, height=700)
o3d.visualization.draw_plotly([input_pcd], width=900, height=700)
o3d.visualization.draw_plotly([recon_mesh], width=900, height=700)

gt_mesh_metrics = mesh_metrics(gt_mesh)
oracle_mesh_metrics = mesh_metrics(oracle_mesh)
recon_mesh_metrics = mesh_metrics(recon_mesh)
print("Ground truth mesh")
print(gt_mesh_metrics)
print("oracle mesh")
print(oracle_mesh_metrics)
print("Reconstructed mesh")
print(recon_mesh_metrics)

print("oracle to ground truth comparison")
print(mesh2mesh_metrics(oracle_mesh_metrics, gt_mesh_metrics))
print("\nreconstruction to ground truth comparison")
print(mesh2mesh_metrics(recon_mesh_metrics, gt_mesh_metrics))
print(f"SPR Time: {spr_time}")


Ground truth mesh
mesh_metrics:
  mesh: TriangleMesh with 2142 points and 4296 triangles.
  n_vertices: 2142
  n_triangles: 4296
  genus: 4.0
  topologically_watertight: None
  o3d_watertight: False
  orientable: True
  self_intersecting: True
  edge_manifold: None
  vertex_manifold: True
  n_components: 1
  largest_cc_area: 2.729117256339001
  largest_cc_area_percentage: 1.0
  largest_cc_o3d_watertight: False
  largest_cc_orientable: True
  largest_cc_self_intersecting: True
  largest_cc_edge_manifold_closed: True
  largest_cc_edge_manifold_open: False
  largest_cc_vertex_manifold: True
  largest_cc_genus: 4.0
  edge_manifold_open: False
  edge_manifold_closed: True
oracle mesh
mesh_metrics:
  mesh: TriangleMesh with 84034 points and 168072 triangles.
  n_vertices: 84034
  n_triangles: 168072
  genus: 4.0
  topologically_watertight: None
  o3d_watertight: False
  orientable: True
  self_intersecting: True
  edge_manifold: None
  vertex_manifold: True
  n_components: 3
  largest_cc_are